In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph

class AgentState(TypedDict):
    file_path: str
    transcript: str
    outline: str
    title: str
    database_id: str

graph_builder = StateGraph(AgentState)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders.parsers.audio import OpenAIWhisperParser


llm = ChatOpenAI(model="gpt-4o", temperature=0)
audio_parser = OpenAIWhisperParser()


In [ ]:
from langchain_core.documents.base import Blob
from langchain_community.document_loaders.parsers.audio import OpenAIWhisperParser


def extract_transcript(state: AgentState) -> AgentState:
  file_path = state["file_path"]
  audio_blob = Blob(path=file_path)
  documents = audio_parser.lazy_parse(audio_blob)
  transcript = ""
  for doc in documents:
    transcript += doc.page_content
  return {"transcript": transcript}

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o", temperature=0)

def generate_outline(state: AgentState) -> AgentState:
  transcript = state["transcript"]
  outline_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that generates an outline for a transcript. Make sure to use Korean when you generate the outline."),
    ("user", "Generate an outline for the following transcript: {transcript}")
  ])
  outline_chain = outline_prompt | llm | StrOutputParser()
  outline = outline_chain.invoke({"transcript": transcript})
  return {"outline": outline}

In [ ]:
import requests
import os

def upload_to_notion(state: AgentState) -> AgentState:
    
    database_id = state['database_id']  
    title = state['title']             
    outline = state['outline']          
    
    notion_api_key = os.getenv("NOTION_API_KEY")
  
    headers = {
        'Authorization': f'Bearer {notion_api_key}',
        'Content-Type': 'application/json',
        'Notion-Version': '2022-06-28'  
    }
    
    data = {
        'parent': {'database_id': database_id}, 
        'properties': {
            'Title': {'title': [{'text': {'content': title}}]},  
        },
        'children': [
            {
                'object': 'block',
                'type': 'paragraph',
                'paragraph': {'rich_text': [{'type': 'text', 'text': {'content': outline}}]}, 
            }
        ]
    }
    
    response = requests.post(
        'https://api.notion.com/v1/pages',
        headers=headers,
        json=data
    )
    
    print(response.json())
    return {}

In [ ]:
graph_builder.add_node(extract_transcript)
graph_builder.add_node(generate_outline)
graph_builder.add_node(upload_to_notion)

In [ ]:
from langgraph.graph import START, END

graph_builder.add_edge(START, 'extract_transcript')
graph_builder.add_edge('extract_transcript', 'generate_outline')
graph_builder.add_edge('generate_outline', 'upload_to_notion')
graph_builder.add_edge('upload_to_notion', END)

graph = graph_builder.compile()

In [ ]:
graph

In [ ]:
graph.invoke({"file_path": "your_video_path", "database_id": "your_notion_database_id", "title": "your_title"})